# 05 · Boosting desde cero: AdaBoost y gradient boosting

**Módulo 4 · Sesión 11** — Boosting e interpretabilidad

## Objetivos

Bagging y Random Forest (sesión 10) promedian árboles **independientes** entrenados en
paralelo. Boosting hace lo contrario: entrena árboles **en secuencia**, cada uno dedicado a
corregir lo que los anteriores hicieron mal. Este notebook construye las dos versiones
clásicas a mano y las valida contra `scikit-learn`:

1. **AdaBoost**: repesar los puntos mal clasificados para que el siguiente tocón se ocupe
   de ellos. Ver cómo cambian los pesos y la frontera ronda a ronda.
2. **Gradient boosting** para regresión: ajustar cada árbol a los **residuales** del
   ensamble anterior — sobre la misma $y = \sin(2\pi x) + \varepsilon$ del módulo 3.
3. Medir lo que distingue a boosting de bagging: **sí sobreajusta** con el número de
   rondas, y la tasa de aprendizaje (*shrinkage*) es la perilla que lo controla.
4. Gradient boosting para **clasificación**: el residual es $y - \hat{p}$, el mismo
   gradiente de la regresión logística del notebook 01.
5. Qué añaden XGBoost y LightGBM sobre esta idea, y cuánto más rápidos son.

La teoría está en `05-boosting.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `xgboost`, `lightgbm`.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import average_precision_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from xgboost import XGBClassifier

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. AdaBoost: repesar los errores

Mismos datos en 2D del notebook 03 (dos medias lunas con ruido). El modelo base es el más
débil posible: un **tocón** (*stump*), un árbol de profundidad 1 — una sola pregunta.

AdaBoost (Freund y Schapire, 1997) mantiene un peso $w_i$ por observación. En cada ronda
$m$:

1. Ajusta un tocón $h_m$ **ponderando** cada punto por $w_i$.
2. Calcula su error ponderado $\varepsilon_m = \sum_i w_i \cdot [h_m(x_i) \neq y_i]$.
3. Le asigna un peso en el ensamble $\alpha_m = \frac{1}{2}\log\frac{1-\varepsilon_m}{\varepsilon_m}$:
   cuanto mejor el tocón, más voz.
4. **Sube** el peso de los puntos que $h_m$ falló ($w_i \leftarrow w_i e^{\alpha_m}$), baja
   el de los que acertó, y normaliza.

La predicción final es el voto ponderado $\text{signo}\left(\sum_m \alpha_m h_m(x)\right)$,
con etiquetas en $\{-1, +1\}$.

In [ ]:
X, y01 = make_moons(n_samples=600, noise=0.32, random_state=SEMILLA)
X_train, X_test, y01_train, y01_test = train_test_split(X, y01, test_size=0.5, random_state=SEMILLA)
y_train, y_test = 2 * y01_train - 1, 2 * y01_test - 1  # etiquetas en {-1, +1}


def adaboost_ajustar(X, y, rondas):
    n = len(y)
    w = np.full(n, 1 / n)
    tocones, alphas, historial_pesos = [], [], []
    for _ in range(rondas):
        h = DecisionTreeClassifier(max_depth=1, random_state=SEMILLA).fit(X, y, sample_weight=w)
        pred = h.predict(X)
        error = np.sum(w * (pred != y))
        alpha = 0.5 * np.log((1 - error) / max(error, 1e-12))
        w = w * np.exp(-alpha * y * pred)  # sube donde y·pred = -1 (fallo), baja donde acierta
        w = w / w.sum()
        tocones.append(h)
        alphas.append(alpha)
        historial_pesos.append(w.copy())
    return tocones, np.array(alphas), historial_pesos


def adaboost_puntuacion(tocones, alphas, X, hasta=None):
    hasta = hasta or len(tocones)
    return sum(a * h.predict(X) for h, a in zip(tocones[:hasta], alphas[:hasta]))


tocones, alphas, historial_pesos = adaboost_ajustar(X_train, y_train, rondas=200)
print(f"Error ponderado del primer tocón: {np.mean(tocones[0].predict(X_train) != y_train):.3f}   alpha_1 = {alphas[0]:.3f}")
print(f"alphas de las rondas 1, 10, 100, 200: {alphas[[0, 9, 99, 199]].round(3)}")

### Los pesos, ronda a ronda

El tamaño de cada punto es su peso $w_i$ en esa ronda. Los puntos difíciles —los que están
del lado equivocado de la frontera— acumulan peso, y el siguiente tocón se ve obligado a
atenderlos.

In [ ]:
def graficar_frontera(puntuar, X, y, eje, titulo, pesos=None):
    g1, g2 = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.75, 2.25, 300))
    z = np.sign(puntuar(np.c_[g1.ravel(), g2.ravel()])).reshape(g1.shape)
    eje.contourf(g1, g2, z, levels=[-1.5, 0, 1.5], colors=["C0", "C1"], alpha=0.25)
    tam = 12 if pesos is None else 4000 * pesos
    eje.scatter(X[y == -1, 0], X[y == -1, 1], c="C0", s=tam if pesos is None else tam[y == -1])
    eje.scatter(X[y == 1, 0], X[y == 1, 1], c="C1", s=tam if pesos is None else tam[y == 1])
    eje.set_title(titulo)
    eje.set_xticks([])
    eje.set_yticks([])


fig, ejes = plt.subplots(1, 4, figsize=(15, 3.6))
for eje, m in zip(ejes, [1, 2, 3, 10]):
    pesos_previos = None if m == 1 else historial_pesos[m - 2]
    graficar_frontera(lambda Z: tocones[m - 1].predict(Z), X_train, y_train, eje,
                      f"tocón {m} (pesos con los que se ajustó)", pesos=pesos_previos)
plt.tight_layout()
plt.show()

fig, ejes = plt.subplots(1, 4, figsize=(15, 3.6))
for eje, m in zip(ejes, [1, 5, 25, 200]):
    graficar_frontera(lambda Z, m=m: adaboost_puntuacion(tocones, alphas, Z, hasta=m), X_train, y_train, eje, f"ensamble tras {m} rondas")
plt.tight_layout()
plt.show()

Cada tocón solo puede trazar una línea vertical u horizontal, pero el voto ponderado de
200 de ellos dibuja las dos medias lunas. A diferencia de bagging, los tocones **no son
intercambiables**: el tocón 3 solo tiene sentido después de los tocones 1 y 2.

### Error de entrenamiento y de prueba por ronda, y comparación con `scikit-learn`

In [ ]:
rondas = np.arange(1, 201)
err_train = [np.mean(np.sign(adaboost_puntuacion(tocones, alphas, X_train, m)) != y_train) for m in rondas]
err_test = [np.mean(np.sign(adaboost_puntuacion(tocones, alphas, X_test, m)) != y_test) for m in rondas]

ada_sk = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200, learning_rate=1.0, random_state=SEMILLA)
ada_sk.fit(X_train, y_train)
err_test_sk = [1 - acc for acc in [np.mean(p == y_test) for p in ada_sk.staged_predict(X_test)]]

plt.figure(figsize=(7, 4))
plt.plot(rondas, err_train, label="entrenamiento (a mano)")
plt.plot(rondas, err_test, label="prueba (a mano)")
plt.plot(rondas, err_test_sk, "--", label="prueba (scikit-learn)")
plt.xlabel("Rondas")
plt.ylabel("Error de clasificación")
plt.title("AdaBoost con tocones")
plt.legend()
plt.show()

print(f"Error de prueba tras 200 rondas — a mano: {err_test[-1]:.3f}   scikit-learn: {err_test_sk[-1]:.3f}")
print(f"Predicciones idénticas a scikit-learn en prueba: {np.mean(np.sign(adaboost_puntuacion(tocones, alphas, X_test)) == ada_sk.predict(X_test)):.3f}")

> `AdaBoostClassifier` implementa SAMME, la generalización multiclase; con dos clases y
> `learning_rate=1.0` se reduce al algoritmo de arriba, y las predicciones coinciden.

El error de entrenamiento sigue bajando mucho después de que el de prueba se estabiliza:
boosting **puede sobreajustar** con las rondas, aunque AdaBoost con tocones lo hace
despacio. Fíjese también en que el error de prueba (0.12) es el mismo que alcanzó bagging
con árboles profundos en el notebook 03 — con modelos base de una sola pregunta. Bagging
reduce la varianza de modelos complejos; boosting reduce el **sesgo** de modelos simples.

## 2. Gradient boosting: ajustar los residuales

AdaBoost repesa puntos; **gradient boosting** (Friedman, 2001) reformula la idea de manera
más general: el ensamble $F_m(x)$ se construye sumando árboles, y cada árbol nuevo se
ajusta al **gradiente negativo de la pérdida** respecto a la predicción actual. Para la
pérdida cuadrática, ese gradiente es simplemente el **residual** $y_i - F_{m-1}(x_i)$:

$$
F_0(x) = \bar{y}, \qquad
F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x), \quad h_m \approx y - F_{m-1}
$$

$\nu \in (0, 1]$ es la **tasa de aprendizaje** (*shrinkage*): cada árbol corrige solo una
fracción del residual, lo que deja trabajo para los siguientes y regulariza.

Sobre la misma función del módulo 3 —$y = \sin(2\pi x) + \varepsilon$—, para poder comparar
contra la verdad.

In [ ]:
def funcion_verdadera(x):
    return np.sin(2 * np.pi * x)


n_reg = 200
x_reg = np.sort(rng.uniform(0, 1, n_reg))
y_reg = funcion_verdadera(x_reg) + rng.normal(0, 0.3, n_reg)
x_reg_test = np.sort(rng.uniform(0, 1, 1000))
y_reg_test = funcion_verdadera(x_reg_test) + rng.normal(0, 0.3, 1000)
Xr, Xr_test = x_reg.reshape(-1, 1), x_reg_test.reshape(-1, 1)


def gb_regresion_ajustar(X, y, rondas, tasa, profundidad=2):
    F = np.full(len(y), y.mean())
    arboles = []
    for _ in range(rondas):
        residual = y - F
        h = DecisionTreeRegressor(max_depth=profundidad, random_state=SEMILLA).fit(X, residual)
        F = F + tasa * h.predict(X)
        arboles.append(h)
    return y.mean(), arboles


def gb_regresion_predecir(F0, arboles, X, tasa, hasta=None):
    hasta = hasta or len(arboles)
    return F0 + tasa * sum(h.predict(X) for h in arboles[:hasta])


TASA = 0.1
F0, arboles = gb_regresion_ajustar(Xr, y_reg, rondas=500, tasa=TASA)

fig, ejes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
malla = np.linspace(0, 1, 400).reshape(-1, 1)
for eje, m in zip(ejes, [1, 5, 50, 500]):
    eje.scatter(x_reg, y_reg, s=8, color="gray", alpha=0.6)
    eje.plot(malla, funcion_verdadera(malla), "k--", lw=1, label="verdad")
    eje.plot(malla, gb_regresion_predecir(F0, arboles, malla, TASA, hasta=m), color="C3", lw=1.5, label=f"F_{m}")
    eje.set_title(f"{m} árbol{'es' if m > 1 else ''} (ν = {TASA})")
    eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

Con un árbol de profundidad 2 y $\nu = 0.1$, $F_1$ apenas se mueve de la media; $F_5$ ya
tiene la forma; $F_{50}$ es una buena aproximación; $F_{500}$ empieza a seguir el ruido. Lo
vemos en la curva de error:

In [ ]:
rondas_reg = np.arange(1, 501)
mse_train = [mean_squared_error(y_reg, gb_regresion_predecir(F0, arboles, Xr, TASA, m)) for m in rondas_reg]
mse_test = [mean_squared_error(y_reg_test, gb_regresion_predecir(F0, arboles, Xr_test, TASA, m)) for m in rondas_reg]

gb_sk = GradientBoostingRegressor(n_estimators=500, learning_rate=TASA, max_depth=2, random_state=SEMILLA).fit(Xr, y_reg)
mse_test_sk = [mean_squared_error(y_reg_test, p) for p in gb_sk.staged_predict(Xr_test)]

plt.figure(figsize=(7, 4))
plt.plot(rondas_reg, mse_train, label="entrenamiento (a mano)")
plt.plot(rondas_reg, mse_test, label="prueba (a mano)")
plt.plot(rondas_reg, mse_test_sk, "--", label="prueba (scikit-learn)")
plt.axhline(0.3**2, color="gray", ls=":", label="ruido irreducible (σ² = 0.09)")
plt.xlabel("Rondas")
plt.ylabel("MSE")
plt.legend()
plt.title("Gradient boosting sí sobreajusta con las rondas")
plt.show()

mejor_m = int(np.argmin(mse_test)) + 1
print(f"MSE de prueba mínimo: {min(mse_test):.4f} en la ronda {mejor_m}; en la ronda 500: {mse_test[-1]:.4f}")
print(f"Máxima diferencia entre la predicción a mano y la de scikit-learn: {np.abs(gb_regresion_predecir(F0, arboles, Xr_test, TASA) - gb_sk.predict(Xr_test)).max():.2e}")

Dos cosas que bagging **no** hacía:

- El error de prueba tiene un **mínimo** y después sube: cada árbol adicional sigue
  ajustando residuales que, pasado cierto punto, son puro ruido. El número de rondas es un
  hiperparámetro de complejidad, no un "más es mejor".
- Las predicciones a mano coinciden con `GradientBoostingRegressor` hasta la precisión
  numérica: el algoritmo es exactamente este.

### La tasa de aprendizaje

$\nu$ y el número de rondas se compensan: con $\nu$ menor hacen falta más árboles para
llegar al mismo punto. A cambio, el mínimo suele ser algo más bajo y, sobre todo, la curva
es **más plana** alrededor de él: equivocarse en el número de rondas cuesta mucho menos.

In [ ]:
plt.figure(figsize=(7, 4))
resumen_tasa = []
for tasa in [1.0, 0.3, 0.1, 0.03]:
    F0_t, arboles_t = gb_regresion_ajustar(Xr, y_reg, rondas=1000, tasa=tasa)
    mse_t = [mean_squared_error(y_reg_test, gb_regresion_predecir(F0_t, arboles_t, Xr_test, tasa, m)) for m in range(1, 1001)]
    plt.plot(range(1, 1001), mse_t, label=f"ν = {tasa}")
    resumen_tasa.append({"ν": tasa, "MSE mínimo": min(mse_t), "ronda del mínimo": int(np.argmin(mse_t)) + 1, "MSE en 1000": mse_t[-1]})
plt.xscale("log")
plt.ylim(0.08, 0.2)
plt.xlabel("Rondas (escala log)")
plt.ylabel("MSE de prueba")
plt.legend()
plt.title("Tasa de aprendizaje: más lento, pero más bajo")
plt.show()
print(pd.DataFrame(resumen_tasa).round(4).to_string(index=False))

Con $\nu = 1$ (sin *shrinkage*) el mínimo llega en la ronda 3 y es el peor (0.1135). Con
$\nu \leq 0.3$ los mínimos son prácticamente iguales (0.109), pero la tolerancia al exceso de
rondas no: en la ronda 1000, $\nu = 0.3$ ha subido a 0.19 y $\nu = 0.03$ solo a 0.135. La
receta práctica que sale de aquí, y que XGBoost y LightGBM heredan: **tasa baja, muchas
rondas, y parar por validación** (*early stopping*) en vez de adivinar el número.

## 3. Gradient boosting para clasificación: el residual es $y - \hat{p}$

¿Qué es "el residual" cuando $y$ es 0/1? Se aplica la misma receta con la pérdida
correcta. El ensamble $F_m(x)$ produce **log-momios**; la probabilidad es
$\hat{p} = \sigma(F_m(x))$; la pérdida es la entropía cruzada del notebook 01; y su gradiente
negativo respecto a $F$ es, exactamente como allí,

$$
-\frac{\partial \mathcal{L}}{\partial F(x_i)} = y_i - \hat{p}_i
$$

Cada árbol de regresión se ajusta a $y - \hat{p}$, y se suma (por $\nu$) a los log-momios.
Es la regresión logística del notebook 01 con los árboles haciendo el papel de
$\mathbf{X}\boldsymbol{\beta}$ — de ahí que boosting produzca probabilidades razonablemente
calibradas, a diferencia de Random Forest.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


def gb_clasificacion_ajustar(X, y, rondas, tasa, profundidad=2):
    p0 = y.mean()
    F = np.full(len(y), np.log(p0 / (1 - p0)))  # log-momios de la prevalencia
    arboles = []
    for _ in range(rondas):
        residual = y - sigmoide(F)
        h = DecisionTreeRegressor(max_depth=profundidad, random_state=SEMILLA).fit(X, residual)
        F = F + tasa * h.predict(X)
        arboles.append(h)
    return np.log(p0 / (1 - p0)), arboles


def gb_clasificacion_proba(F0, arboles, X, tasa):
    return sigmoide(F0 + tasa * sum(h.predict(X) for h in arboles))


F0_c, arboles_c = gb_clasificacion_ajustar(X_train, y01_train, rondas=200, tasa=0.1)
p_mano = gb_clasificacion_proba(F0_c, arboles_c, X_test, 0.1)

gbc_sk = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=2, random_state=SEMILLA).fit(X_train, y01_train)
p_sk = gbc_sk.predict_proba(X_test)[:, 1]

print(f"Accuracy en prueba — a mano: {np.mean((p_mano >= 0.5) == y01_test):.3f}   scikit-learn: {np.mean((p_sk >= 0.5) == y01_test):.3f}")
print(f"AP en prueba       — a mano: {average_precision_score(y01_test, p_mano):.3f}   scikit-learn: {average_precision_score(y01_test, p_sk):.3f}")
print(f"Correlación entre las probabilidades de ambos: {np.corrcoef(p_mano, p_sk)[0, 1]:.4f}")

> Aquí las probabilidades no coinciden dígito a dígito con `scikit-learn`, y la razón es
> instructiva: la implementación de arriba da un paso de **gradiente** (la hoja vale el
> residual medio), mientras que `GradientBoostingClassifier` da un paso de **Newton** en
> cada hoja —divide por la curvatura $\sum \hat{p}_i(1 - \hat{p}_i)$—, que converge más
> rápido. XGBoost generaliza justo eso a cualquier pérdida.

## 4. Qué añaden XGBoost y LightGBM, y cuánto más rápidos son

Las dos librerías dominantes implementan gradient boosting con árboles, más:

| Idea | XGBoost (2016) | LightGBM (2017) |
|---|---|---|
| Paso de **Newton** (segunda derivada) en cada hoja | Sí | Sí |
| **Regularización** explícita del valor de las hojas ($\lambda$, $\gamma$) | Sí | Sí |
| Submuestreo de filas y columnas por árbol (como Random Forest) | Sí | Sí |
| **Histogramas**: discretizar cada variable en ≈256 cubetas antes de buscar el umbral | Opcional (`tree_method="hist"`) | Siempre |
| Crecimiento **por hoja** (*leaf-wise*): expande la hoja con mayor ganancia, no nivel a nivel | No (por nivel) | Sí (`num_leaves`) |
| Categóricas nativas, sin *one-hot* | Experimental | Sí |

La aceleración viene sobre todo de los histogramas: la búsqueda de umbral del notebook 03
era $O(n \log n)$ por variable; con 256 cubetas es $O(n)$ para construir el histograma y
$O(256)$ para buscar. Lo medimos sobre 30 000 filas y 20 variables, con 200 árboles de
profundidad 6 en todos los casos (`HistGradientBoostingClassifier` es la versión con
histogramas de `scikit-learn`, inspirada en LightGBM).

In [ ]:
X_grande, y_grande = make_classification(n_samples=30_000, n_features=20, n_informative=10, random_state=SEMILLA)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(X_grande, y_grande, test_size=0.2, random_state=SEMILLA)

implementaciones = {
    "GradientBoostingClassifier (exacto)": GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=SEMILLA),
    "HistGradientBoostingClassifier": HistGradientBoostingClassifier(max_iter=200, max_depth=6, learning_rate=0.1, random_state=SEMILLA),
    "XGBClassifier (hist)": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, tree_method="hist", random_state=SEMILLA, n_jobs=4, verbosity=0),
    "LGBMClassifier": LGBMClassifier(n_estimators=200, max_depth=6, num_leaves=63, learning_rate=0.1, random_state=SEMILLA, n_jobs=4, verbose=-1),
}
filas = []
for nombre, modelo in implementaciones.items():
    t0 = time.time()
    modelo.fit(Xg_train, yg_train)
    t_ajuste = time.time() - t0
    filas.append({"implementación": nombre, "ajuste (s)": t_ajuste, "AP en prueba": average_precision_score(yg_test, modelo.predict_proba(Xg_test)[:, 1])})
print(pd.DataFrame(filas).round(3).to_string(index=False))

Misma idea, misma AP, y **dos órdenes de magnitud** de diferencia en tiempo entre la
implementación exacta y las de histogramas. Esa diferencia es la que hace posible afinar
hiperparámetros con Optuna en minutos en vez de horas, que es lo que el notebook 06 hace
sobre Wine Quality.

## Resumen

| Concepto | Lo que se vio |
|---|---|
| AdaBoost | Repesar los fallos; voto ponderado de tocones dibuja las dos medias lunas; coincide con `scikit-learn` |
| Gradient boosting | Cada árbol ajusta el residual (gradiente negativo de la pérdida) del ensamble anterior; coincide con `scikit-learn` hasta precisión numérica |
| Rondas | A diferencia de bagging, **hay un mínimo**: más árboles acaban ajustando ruido |
| Tasa de aprendizaje | Menor $\nu$ → más rondas, mínimo similar y curva mucho más plana; receta: $\nu$ baja + early stopping |
| Clasificación | El residual es $y - \hat{p}$: el gradiente de la logística del notebook 01, con árboles en vez de $\mathbf{X}\boldsymbol{\beta}$ |
| XGBoost / LightGBM | Newton en las hojas, regularización, histogramas, crecimiento por hoja: ~100× más rápido con la misma AP |